In [0]:
dbutils.widgets.text("p_environment", "production")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../Includes/configuration"

In [0]:
%run "../Includes/common_functions"

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

In [0]:
movie_company_schema = StructType(fields= [
                       StructField("movieId", IntegerType(), True),
                       StructField("companyId", IntegerType(), True)
])

In [0]:
movie_company_df = spark.read \
                   .option("header", True) \
                   .schema(movie_company_schema) \
                   .csv(f"{bronze_folder_path}/{v_file_date}/movie_company")

In [0]:
display(movie_company_df)

movieId,companyId
168259,33
168259,333
168259,3341
168259,40890
168259,6452
168259,7154
168259,86352
168259,86655
168259,87857
168259,87858


In [0]:
movie_company_df.count()

1676

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
movie_company_final_df = add_ingestion_date(movie_company_df) \
                            .withColumnsRenamed({"movieId": "movie_Id", "companyId": "company_Id"}) \
                            .withColumn("environment", lit(v_environment)) \
                            .withColumn("file_date", lit(v_file_date))

In [0]:
display(movie_company_final_df)

movie_Id,company_Id,ingestion_date,environment,file_date
168259,33,2026-09-11T04:39:12.743345Z,production,2024-12-30
168259,333,2026-09-11T04:39:12.743345Z,production,2024-12-30
168259,3341,2026-09-11T04:39:12.743345Z,production,2024-12-30
168259,40890,2026-09-11T04:39:12.743345Z,production,2024-12-30
168259,6452,2026-09-11T04:39:12.743345Z,production,2024-12-30
168259,7154,2026-09-11T04:39:12.743345Z,production,2024-12-30
168259,86352,2026-09-11T04:39:12.743345Z,production,2024-12-30
168259,86655,2026-09-11T04:39:12.743345Z,production,2024-12-30
168259,87857,2026-09-11T04:39:12.743345Z,production,2024-12-30
168259,87858,2026-09-11T04:39:12.743345Z,production,2024-12-30


In [0]:
# overwrite_partition("movie_silver", "movies_companies", "file_date", v_file_date)  

In [0]:
merge_delta_lake_2(movie_company_final_df, "movie_silver", "movies_companies", "movie_Id", "company_Id", "file_date")

In [0]:
# movie_company_final_df.write.mode("append").partitionBy("file_date").format("delta").saveAsTable("movie_silver.movies_companies")

In [0]:
display(spark.read.table("movie_silver.movies_companies"))

movie_Id,company_Id,ingestion_date,environment,file_date
21345,5552,2026-09-11T04:38:38.226067Z,production,2024-12-23
21349,306,2026-09-11T04:38:38.226067Z,production,2024-12-23
21349,711,2026-09-11T04:38:38.226067Z,production,2024-12-23
21355,14,2026-09-11T04:38:38.226067Z,production,2024-12-23
21355,5,2026-09-11T04:38:38.226067Z,production,2024-12-23
21413,14,2026-09-11T04:38:38.226067Z,production,2024-12-23
21413,7161,2026-09-11T04:38:38.226067Z,production,2024-12-23
21512,3970,2026-09-11T04:38:38.226067Z,production,2024-12-23
21512,4255,2026-09-11T04:38:38.226067Z,production,2024-12-23
21512,68152,2026-09-11T04:38:38.226067Z,production,2024-12-23


In [0]:
%sql
SELECT file_date, COUNT(1)
FROM movie_silver.movies_companies
GROUP BY file_date;

file_date,count(1)
2024-12-16,7996
2024-12-23,3998
2024-12-30,1676


In [0]:
%sql
DESCRIBE EXTENDED movie_silver.movies_companies;

col_name,data_type,comment
movie_Id,int,null
company_Id,int,null
ingestion_date,timestamp,null
environment,string,null
file_date,string,null
# Partition Information,,
# col_name,data_type,comment
file_date,string,null
,,
# Delta Statistics Columns,,


In [0]:
dbutils.notebook.exit("Success")